# 06 · Second-order optimization (closed-form curvature)

Because `σ′` and `σ″` are closed form, the **parameter** Hessian / Gauss–Newton
Fisher of a one-layer field can be assembled algebraically — no autodiff. That
makes Newton / natural-gradient / KFAC steps cheap.

`omnibias-curvature` exposes `one_layer_param_hessian`,
`mse_gauss_newton_fisher`, `mse_newton_step`, and `kfac_kron_factors`. Here we
fit a one-layer field by Gauss–Newton and watch it converge in a handful of
steps, versus first-order gradient descent.

In [ ]:
import os, sys
os.environ["JAX_ENABLE_X64"] = "true"
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, ACCENT, GOOD, PRIMARY
set_style()

from omnibias.jax import get_activation
from omnibias.curvature import (
    mse_gauss_newton_fisher, mse_newton_step, kfac_kron_factors,
)

NAME = "tanh"
sig = get_activation(NAME).forward

def predict(X, W, beta, c, b):
    return b + sig(X @ W.T + beta) @ c

def mse(X, Y, W, beta, c, b):
    return jnp.mean((predict(X, W, beta, c, b) - Y) ** 2)

## Data + initialisation

Synthetic regression: a random one-layer "teacher" generates the targets, and
we fit a fresh "student" of the same shape.

In [ ]:
key = jax.random.PRNGKey(0)
B, D, H = 256, 4, 6
kx, kt, kw, kb, kc = jax.random.split(key, 5)

X = jax.random.normal(kx, (B, D))
Wt = jax.random.normal(kt, (H, D)); betat = jax.random.normal(kb, (H,)); ct = jax.random.normal(kc, (H,))
Y = predict(X, Wt, betat, ct, 0.0)  # teacher targets

# Student init
W = 0.3 * jax.random.normal(kw, (H, D))
beta = jnp.zeros(H); c = 0.3 * jax.random.normal(kc, (H,)); b = jnp.array(0.0)
print("start MSE =", float(mse(X, Y, W, beta, c, b)))

## Gauss–Newton vs gradient descent

The closed-form Fisher gives us both the curvature and the gradient. We compare
a Gauss–Newton step against plain gradient descent using the *same* gradient.

In [ ]:
# Gauss-Newton (closed-form Fisher) path
Wn, betan, cn, bn = W, beta, c, b
gn_loss = [float(mse(X, Y, Wn, betan, cn, bn))]
for _ in range(15):
    bn, cn, betan, Wn = mse_newton_step(X, Y, Wn, betan, cn, bn, NAME,
                                        learning_rate=1.0, damping=1e-6)
    gn_loss.append(float(mse(X, Y, Wn, betan, cn, bn)))

# First-order gradient descent using the same closed-form gradient
Wg, betag, cg, bg = W, beta, c, b
gd_loss = [float(mse(X, Y, Wg, betag, cg, bg))]
lr = 0.5
for _ in range(15):
    F, g = mse_gauss_newton_fisher(X, Y, Wg, betag, cg, bg, NAME)
    # unpack flat grad [b, c(H), beta(H), W(H*D)] and descend
    bg = bg - lr * g[0]
    cg = cg - lr * g[1:1 + H]
    betag = betag - lr * g[1 + H:1 + 2 * H]
    Wg = Wg - lr * g[1 + 2 * H:].reshape(H, D)
    gd_loss.append(float(mse(X, Y, Wg, betag, cg, bg)))

fig, ax = plt.subplots()
ax.semilogy(gd_loss, "o-", color=ACCENT, label="gradient descent")
ax.semilogy(gn_loss, "s-", color=GOOD, label="Gauss–Newton (closed-form Fisher)")
ax.set_xlabel("iteration"); ax.set_ylabel("MSE"); ax.set_title("Convergence: 2nd-order vs 1st-order")
ax.legend(); plt.show()
print(f"Gauss-Newton final MSE = {gn_loss[-1]:.2e}  in 15 steps")

## KFAC factors come for free

The closed-form Fisher also yields KFAC Kronecker factors `(A, G)` for the
hidden weight block, the ingredient natural-gradient optimizers need.

In [ ]:
A, G = kfac_kron_factors(X, Wn, betan, cn, bn, NAME)
print("KFAC factors:  A", A.shape, " G", G.shape)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.8))
a1.imshow(A, cmap="cividis"); a1.set_title("A  (input covariance)")
a2.imshow(G, cmap="cividis"); a2.set_title("G  (pre-activation grad cov)")
for a in (a1, a2):
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Takeaway

Closed-form `σ′`/`σ″` make the parameter Fisher / Hessian and KFAC factors
**exact and autodiff-free**, so Gauss–Newton converges in a few steps. This is
the basis for natural-gradient training of omnibias fields (and the FermiNet
KFAC integration on the roadmap).

Next: **[07 · CmbNet (operator-typed CNN)](07_cmbnet_mnist.ipynb)**.